# 📑 PageIndex — Vectorless RAG Crash Course
### Reasoning-based RAG with No Vector DB, No Chunking
**By [@krishnaik06](https://youtube.com/@krishnaik06) | [krishnaik.in/liveclasses](https://krishnaik.in/liveclasses)**

---

## 🧠 What You'll Learn

| # | Topic |
|---|-------|
| 1 | Why Vector RAG fails on professional documents |
| 2 | How PageIndex builds a tree index from a PDF |
| 3 | LLM Tree Search — reasoning over structure |
| 4 | Full end-to-end Vectorless RAG pipeline |
| 5 | Expert-guided retrieval (domain knowledge injection) |
| 6 | Chat API — zero LLM setup |
| 7 | Self-hosted open-source option |

---

## 🔑 Key Concept

> **Traditional RAG** → chunk → embed → cosine similarity → retrieve  
> **PageIndex RAG** → build tree → LLM reasons over tree → retrieve exact sections

**The problem with vector RAG:**  
`Similarity ≠ Relevance`  
A chunk about "market conditions" may score higher than the actual answer section just because it shares more words with your query.


---
## 📦 Section 1: Install & Setup

**What we do here:**
- Install PageIndex SDK + OpenAI
- Load API keys from `.env`
- Initialize both clients

> 🔑 Get your **PageIndex API key** from: https://dash.pageindex.ai/api-keys  
> 🔑 Get your **OpenAI API key** from: https://platform.openai.com


In [2]:
# Install required packages
!pip install -U pageindex groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 2.9 MB/s eta 0:00:00


In [ ]:
# ── Create a .env file (run this once) ──────────────────────────────────────
# Uncomment and fill in your keys, then run this cell ONCE

# env_content = """
# PAGEINDEX_API_KEY=your_pageindex_key_here
# GROQ_API_KEY=your_groq_key_here
# """
# with open(".env", "w") as f:
#     f.write(env_content.strip())
# print("✅ .env file created")

In [ ]:
import os, json, time
# from dotenv import load_dotenv

# load_dotenv()
PAGEINDEX_API_KEY = ''
# GROQ_API_KEY      = ''
GROQ_API_KEY      = ''

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("Groq key loaded:     ", "✅" if GROQ_API_KEY      else "❌ Missing!")

PageIndex key loaded: ✅
Groq key loaded:      ✅


In [5]:
from pageindex import PageIndexClient
from groq import Groq

pi_client   = PageIndexClient(api_key=PAGEINDEX_API_KEY)
groq_client = Groq(api_key=GROQ_API_KEY)

print("✅ PageIndex client ready")
print("✅ Groq client ready")

✅ PageIndex client ready
✅ Groq client ready


---
## 🌲 Section 2: Upload & Index a PDF

**What happens here:**
1. Upload your PDF to the PageIndex cloud
2. PageIndex uses an LLM to read the document structure
3. Builds a hierarchical **tree index** (like a smart Table of Contents)
4. Returns a `doc_id` for all future operations

**Why NO chunking?**  
Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.


In [6]:
# ── Upload your PDF ─────────────────────────────────────────────────────────
# Replace with the path to your PDF file
# Great candidates: Annual reports, research papers, legal docs, textbooks

PDF_PATH = "/content/data science.pdf"   # ← change this

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

📤 Uploading: /content/data science.pdf
✅ Uploaded!
📋 Document ID: pi-cmopdz15t01ad01qrynp5qtls
   (Save this ID — you'll use it throughout the notebook)


In [7]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")

    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break

    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: processing
   Status: processing
   Status: processing
   Status: processing
   Status: completed

✅ Tree index ready!


---
## 🔍 Section 3: Inspect the Tree Structure

**What the tree looks like:**

```
Document
├── Introduction (pages 1-3)
│   └── Background (pages 1-2)
├── Financial Stability (pages 21-31)
│   ├── Monitoring Vulnerabilities (pages 22-28)
│   └── International Cooperation (pages 28-31)
└── Conclusion (pages 45-47)
```

Each node has:
- `node_id` — unique ID used during retrieval
- `title` — section heading
- `page_index` — page number in original PDF
- `text` — section summary (when `node_summary=True`)
- `nodes` — child sections (nested)

**This structure is what the LLM reasons over during retrieval.**


In [8]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 3

🌲 Raw tree (first node):
{
  "title": "Preface",
  "node_id": "0000",
  "page_index": 1,
  "summary": "DATA SCIENCE INTERVIEW\n\n![img-0.jpeg](img-0.jpeg)\n\nDSI ACE PREP\n\nDATA SCIENCE\nINTERVIEW\nGUIDE\nACE-PREP\n",
  "text": "DATA SCIENCE INTERVIEW\n\n![img-0.jpeg](img-0.jpeg)\n\nDSI ACE PREP\n\nDATA SCIENCE\nINTERVIEW\nGUIDE\nACE-PREP\n"
}


In [9]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Preface  (p.1)
[0001] ABOUT THE AUTHOR  (p.3)
[0002] ACE-PREP  (p.3)
  └─ [0003] MASTERSHIP BOOKS  (p.5)
  └─ [0004] CONTENTS  (p.9)
  └─ [0005] INTRODUCTION  (p.11)
    └─ [0006] Understanding Data Science: Definitions, Applications, and Core Concepts  (p.11)
    └─ [0007] Core Data Science Concepts and Methodologies  (p.15)
    └─ [0008] Feature Selection, Data Handling, and Model Evaluation Techniques  (p.19)
    └─ [0009] Data Science Fundamentals: Metrics, Balancing, Careers, and Statistics  (p.22)
  └─ [0010] MASTERING THE BASICS  (p.25)
    └─ [0011] Core Statistics and Probability Concepts for Interviews  (p.25)
    └─ [0012] Foundational Concepts: Probability and Linear Algebra  (p.29)
    └─ [0013] Probability, Linear Algebra, and Python Fundamentals  (p.33)
  └─ [0014] PYTHON  (p.37)
  └─ [0015] PANDA  (p.45)
    └─ [0016] Pandas: Core Concepts and Operations  (p.45)
    └─ [0017] NumPy: Core Concepts, Data Handling, and Array Operations  (

In [10]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 129
   Each node = one retrievable section of the document


---
## 🧠 Section 4: LLM Tree Search — The Core of PageIndex

**This is where PageIndex fundamentally differs from vector RAG.**

### Vector RAG retrieval:
```
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
```
*Problem: finds what's similar, not what's relevant*

### PageIndex retrieval:
```
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
```
*Advantage: LLM understands document structure, context, and intent*

**The LLM acts like a human expert scanning a Table of Contents.**


In [32]:
def llm_tree_search(query: str, tree: list, model: str = "llama3-8b-8192") -> dict:
    """
    Core PageIndex retrieval using Groq (switched to 8b model to avoid rate limits):
    Sends the query + document tree to an LLM.
    """
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    compressed_tree = compress(tree)

    prompt = f"""You are given a query and a document's tree structure.
Identify which node IDs most likely contain the answer. Reply ONLY in JSON format.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

{{
  "thinking": "<reasoning>",
  "node_list": ["node_id1"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)

In [13]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What are the core topics covered in the Statistics and Probability section for interviews?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What are the core topics covered in the Statistics and Probability section for interviews?

🧠 LLM Reasoning:
The query asks about the core topics covered in the Statistics and Probability section for interviews. Looking at the document tree, we find that the section 'MASTERING THE BASICS' with node_id '0010' has a child node 'Core Statistics and Probability Concepts for Interviews' with node_id '0011'. This suggests that node '0011' is likely to contain the information we're looking for.

🎯 Selected Node IDs: ['0011']


---
## ⚙️ Section 5: Full End-to-End RAG Pipeline

**3 steps:**
1. **Tree Search** → LLM picks relevant `node_ids`
2. **Retrieve** → Fetch the actual section content from those nodes  
3. **Generate** → LLM writes a grounded answer with page citations

**What makes this better than vector RAG:**
- Retrieved content has titles + page numbers (traceable)
- LLM can cite exactly *which section* the answer comes from
- No hallucination from irrelevant chunks


In [14]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [15]:
def generate_answer(query: str, nodes: list, model: str = "llama3-8b-8192") -> str:
    """
    Takes retrieved nodes as context and generates an answer using Groq 8b.
    """
    if not nodes:
        return "⚠️ No relevant sections found."

    context = "\n\n---\n\n".join([
        f"[Section: '{n['title']}' | Page {n.get('page_index', '?')}]\n{n.get('text', 'N/A')}"
        for n in nodes
    ])

    prompt = f"""Answer the question using ONLY the context. Cite sources.
Question: {query}
Context:
{context}
Answer:"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [16]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:

    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")

    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])

    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")

    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")

    # Step 3: Generate answer
    answer = generate_answer(query, nodes)

    if verbose:
        print(f"\n📝 Answer:\n{answer}")

    return answer

In [17]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What are the common interview questions related to Support Vector Machines (SVM)?",
    tree=pageindex_tree
)

🔍 Query: What are the common interview questions related to Support Vector Machines (SVM)?

🧠 Reasoning: To identify which node IDs most likely contain the answer, we need to analyze the provided document tree structure and look for sections related to Support Vector Machines (SVM) and interview question...
🎯 Retrieved node IDs: ['0023', '0020', '0021', '0022']
📄 Sections found: ['Machine Learning Interview Questions', 'PCA Interview Questions', 'Curse of Dimensionality', 'Support Vector Machine (SVM)']

📝 Answer:
Based on the provided context, the common interview questions related to Support Vector Machines (SVM) are:

1. Could you explain SVM to me? (Page 66, Section: 'Support Vector Machine (SVM)')
2. In light of SVMs, how would you explain Convex Hull? (Page 66, Section: 'Support Vector Machine (SVM)')
3. Should you train a model on a training set with millions of instances and hundreds of features using the primal or dual form of the SVM problem? (Page 66, Section: 'Support Vecto

In [18]:
# ── Test with multiple queries ───────────────────────────────────────────────
test_queries = [
    "What are the core topics in Machine Learning interview questions?",
    "Explain the difference between clustered and non-clustered indexes in SQL.",
    "Summarize the key concepts covered in the Python section.",
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:300]}...")
    print("-" * 55)


Q: What are the core topics in Machine Learning interview questions?
A: According to the provided context, the core topics in Machine Learning interview questions include:

1. **Definition and purpose of Machine Learning**: Understanding the concept of Machine Learning, its types (supervised, unsupervised, reinforcement learning), and its applications.
2. **Supervised L...
-------------------------------------------------------

Q: Explain the difference between clustered and non-clustered indexes in SQL.
A: According to the context, the difference between clustered and non-clustered indexes in SQL is as follows:

Clustered indexes are utilized for quicker data retrieval from databases, whereas reading from non-clustered indexes takes longer. A clustered index changes the way records are stored in a dat...
-------------------------------------------------------

Q: Summarize the key concepts covered in the Python section.
A: Based on the provided context, the key concepts covered in t

---
## 🎓 Section 6: Expert-Guided Retrieval

**The killer feature no one talks about.**

With vector RAG, injecting domain expertise requires **fine-tuning the embedding model** — expensive and time-consuming.

With PageIndex, you just **add rules to the prompt**:

```
"If the query mentions EBITDA → prioritize the MD&A section"
"If the query is about risks  → check Part I, Item 1A"
```

This makes PageIndex instantly adaptable to any domain — finance, legal, medical, technical — without any model training.


In [ ]:
# ── Define domain expert rules ───────────────────────────────────────────────
# These are routing rules that tell the LLM WHERE to look for specific queries.
# Think of it as encoding a senior analyst's institutional knowledge.


In [24]:

# These rules are specifically tailored for the Data Science Interview Guide
FINANCIAL_EXPERT_RULES = """
Expert routing rules for Data Science Interview Guide:
- Statistics & Probability       → Sections [0011], [0012]
- Linear Algebra                 → Sections [0012], [0013]
- Overfitting / Underfitting     → Section [0024]
- Neural Networks / Deep Learning → Sections [0026] to [0031]
- SQL Indexes & Integrity        → Sections [0097], [0108]
- Data Wrangling                 → Section [0119]
"""

print("✅ Expert rules updated for Data Science guide")
print("   The LLM will now use these domain rules to navigate the tree.")

✅ Expert rules updated for Data Science guide
   The LLM will now use these domain rules to navigate the tree.


In [25]:
# ── Expert Routing Rules for Data Science Interview Guide ─────────────────────
DATA_SCIENCE_EXPERT_RULES = """
Route queries to the correct section using these rules:
- Statistics & Probability       → Section [0011], [0012]
- Python Programming basics      → Section [0014]
- Pandas & NumPy operations      → Section [0015], [0016], [0017]
- Machine Learning (General)     → Section [0019], [0020]
- SVM, PCA, Dimensionality       → Section [0021], [0022], [0023]
- Neural Networks & Deep Learning → Section [0026] to [0031]
- SQL & Database Integrity       → Section [0091], [0097], [0108]
- Data Wrangling & Visualization → Section [0119], [0121], [0122]
"""
print("✅ Data Science Expert rules defined")

✅ Data Science Expert rules defined


In [42]:
# ── Expert-guided tree search ────────────────────────────────────────────────

def llm_tree_search_with_expert(
    query: str,
    tree: list,
    expert_rules: str,
    model: str = "llama-3.3-70b-versatile"
) -> dict:
    """
    Same as llm_tree_search() but with domain expert rules injected using 8b model.
    """

    def compress(nodes):
        out = []
        for n in nodes:
            entry = {"node_id": n["node_id"], "title": n["title"],
                     "page": n.get("page_index", "?"),
                     "summary": n.get("text", "")[:150]}
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    prompt = f"""You are a domain expert analyzing a document.
Find all node IDs that most likely contain the answer to the query.
Use the expert routing rules below to guide your reasoning.

Query: {query}

Document Tree:
{json.dumps(compress(tree), indent=2)}

Expert Routing Rules (follow these carefully):
{expert_rules}

Reply ONLY in this JSON format:
{{
  "thinking": "<your reasoning, referencing the expert rules>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = groq_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [49]:
# ── Test expert-guided retrieval ─────────────────────────────────────────────
query = "What are the advanced SQL indexing concepts and data integrity types?"

print(f"🔍 Query: {query}\n")

# With expert rules
print("── With Data Science Expert Rules ──")
guided  = llm_tree_search_with_expert(query, pageindex_tree, DATA_SCIENCE_EXPERT_RULES)
print("Nodes:", guided.get("node_list"))
print("Reasoning:", guided.get("thinking", ""))

🔍 Query: What are the advanced SQL indexing concepts and data integrity types?

── With Data Science Expert Rules ──
Nodes: ['0091', '0097', '0108']
Reasoning: The query about advanced SQL indexing concepts and data integrity types can be solved by referencing the expert routing rules. According to the rules, SQL & Database Integrity related queries should be routed to sections [0091], [0097], and [0108]. These sections cover Unique Constraints, Clustered and Non-Clustered Indexes, and Data Integrity. Therefore, the nodes that most likely contain the answer to the query are the ones corresponding to these sections.


In [50]:
# ── Full expert-guided RAG ───────────────────────────────────────────────────

def expert_rag(query: str, tree: list, rules: str) -> str:
    """Expert-guided end-to-end RAG pipeline."""
    result  = llm_tree_search_with_expert(query, tree, rules)
    nodes   = find_nodes_by_ids(tree, result.get("node_list", []))
    return generate_answer(query, nodes)

# Run it
answer = expert_rag(
    query="Explain Clustered vs Non-Clustered indexes based on the guide",
    tree=pageindex_tree,
    rules=DATA_SCIENCE_EXPERT_RULES
)
print(answer)

According to the guide (Section: '10. What is the difference between a clustered and a non-clustered index?' | Page 89), the differences between a clustered and a non-clustered index are:

- Clustered indexes are utilized for quicker data retrieval from databases, whereas reading from non-clustered indexes takes longer.
- A clustered index changes the way records are stored in a database by sorting rows by the clustered index column. A non-clustered index does not change the way records are stored but instead creates a separate object within a table that points back to the original table rows after searching.

Additionally, as mentioned in (Section: '6. In SQL Server, what is the difference between a clustered and a non-clustered index?' | Page 113), 
- There can only be one clustered index per table, although several non-clustered indexes can be.
- The Clustered Index is quicker than the Non-Clustered Index by a little margin. When a Non-Clustered Index is used, an extra lookup from t

In [ ]:
# Re-loading API Keys
PAGEINDEX_API_KEY = ''
GROQ_API_KEY      = ''

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("Groq key loaded:     ", "✅" if GROQ_API_KEY      else "❌ Missing!")

PageIndex key loaded: ✅
Groq key loaded:      ✅


In [48]:
# Re-initializing Clients
from pageindex import PageIndexClient
from groq import Groq

pi_client   = PageIndexClient(api_key=PAGEINDEX_API_KEY)
groq_client = Groq(api_key=GROQ_API_KEY)

print("✅ PageIndex client refreshed")
print("✅ Groq client refreshed")

✅ PageIndex client refreshed
✅ Groq client refreshed


---
## 💬 Section 7: Chat API — Zero LLM Setup

**When to use this:**
- You don't want to manage OpenAI API calls yourself
- You want a quick Q&A interface over your document
- You're building a chat product and want PageIndex to handle everything

PageIndex provides its own LLM — you just pass a question and `doc_id`.


In [51]:
# ── Single question with Chat API ────────────────────────────────────────────
# No OpenAI key needed — PageIndex runs the LLM internally

question = "What are the main sections and learning objectives of this interview guide?"

response = pi_client.chat_completions(
    messages=[{"role": "user", "content": question}],
    doc_id=doc_id
)

answer = response["choices"][0]["message"]["content"]
print("💬 Chat API Answer:")
print(answer)

💬 Chat API Answer:
Here's an overview of the **Data Science Interview Guide**'s main sections and what each covers:

---

## 📚 Main Sections

### Introduction
- What Data Science means
- Background interview Q&A
- Careers in Data Science

### Chapter 1 – Mastering the Basics
- Statistics, Probability, and Linear Algebra fundamentals

### Chapter 2 – Python
- Interview questions focused on Python programming

### Chapter 3 – Pandas & NumPy
- NumPy interview questions and data manipulation

### Chapter 4 – Machine Learning
- PCA, Curse of Dimensionality, Support Vector Machines (SVM), Overfitting & Underfitting

### Chapter 5 – R Language
- CSV files in R, Confusion Matrix, Random Forest, K-Means Clustering

### Chapter 6 – SQL
- DBMS vs RDBMS, MySQL, Unique Constraints, Clustered/Non-Clustered Indexes, Data Integrity, SQL Cursors

### Chapter 7 – Data Wrangling & Visualization
- Techniques for cleaning and visualizing data

### Chapter 8 – Data Science Interview Extra
- Extra interview 

In [53]:
# ── Multi-turn conversation ───────────────────────────────────────────────────
# Keep the full message history for context across turns

conversation_history = []

def chat_with_doc(user_message: str, doc_id: str) -> str:
    """Chat with a document, maintaining conversation history."""
    global conversation_history

    conversation_history.append({"role": "user", "content": user_message})

    response = pi_client.chat_completions(
        messages=conversation_history,
        doc_id=doc_id
    )

    assistant_reply = response["choices"][0]["message"]["content"]
    conversation_history.append({"role": "assistant", "content": assistant_reply})

    return assistant_reply


# Simulate a 3-turn conversation relevant to the Data Science Guide
questions = [
    "What are the key topics covered for Support Vector Machines (SVM)?",
    "How does the guide explain the difference between Pandas and NumPy?",
    "What is the final advice given in the conclusion section?"
]

for q in questions:
    print(f"\n👤 User: {q}")
    reply = chat_with_doc(q, doc_id)
    print(f"🤖 Assistant: {reply[:400]}...")
    print("-" * 55)


👤 User: What are the key topics covered for Support Vector Machines (SVM)?
🤖 Assistant: Now I can see SVM is in **Chapter 4: Machine Learning**. Let me locate those pages.Here are the key topics covered for **Support Vector Machines (SVM)** in the document (pages 66–68):

---

### 📌 Overview
SVM is introduced as a **supervised machine learning algorithm** used for both **classification and regression** problems, particularly well-suited for **complex but small or medium-sized dataset...
-------------------------------------------------------

👤 User: How does the guide explain the difference between Pandas and NumPy?
🤖 Assistant: Here's how the guide explains the differences between **Pandas** and **NumPy** across Chapter 3 (pages 45–54):

---

### 🔢 NumPy — Numerical Computing Foundation
- Stands for **"Numerical Python"**
- Core structure: **ndarray** — a high-performance, multi-dimensional array that stores values of the **same data type**
- Designed for **scientific computing**: s

---
## 🛠️ Section 8: Self-Hosted Open Source Option

**Use this when:**
- You don't want to send documents to any cloud
- You need full data privacy / on-prem deployment
- You want to inspect or customize the tree-building logic

The open-source repo at https://github.com/VectifyAI/PageIndex lets you run the entire pipeline locally using your own OpenAI key.

**What the CLI does:**
1. Reads your PDF
2. Detects existing Table of Contents (if any)
3. Uses GPT-4o to build the hierarchical tree
4. Saves a `document_name_pageindex.json` alongside your PDF


In [ ]:
# ── Clone the open-source repo ───────────────────────────────────────────────
!git clone https://github.com/VectifyAI/PageIndex.git
%cd PageIndex
!pip install -r requirements.txt

In [ ]:
# ── Create .env for self-hosted mode ─────────────────────────────────────────
# The local runner uses CHATGPT_API_KEY (not OPENAI_API_KEY)

import os
openai_key = os.getenv("OPENAI_API_KEY", "your_key_here")

with open(".env", "w") as f:
    f.write(f"CHATGPT_API_KEY={openai_key}\n")

print("✅ .env created for self-hosted mode")

In [ ]:
# ── Run PageIndex locally on a PDF ───────────────────────────────────────────
# Optional parameters you can customize:
#   --model                  OpenAI model (default: gpt-4o-2024-11-20)
#   --toc-check-pages        Pages to scan for existing TOC (default: 20)
#   --max-pages-per-node     Max pages per tree node (default: 10)
#   --if-add-node-summary    Include summaries in output (yes/no)

PDF_PATH = "/path/to/your/document.pdf"   # ← change this

!python run_pageindex.py \
    --pdf_path {PDF_PATH} \
    --model gpt-4o-2024-11-20 \
    --toc-check-pages 20 \
    --max-pages-per-node 10 \
    --if-add-node-summary yes

In [ ]:
# ── Load locally generated tree ──────────────────────────────────────────────
# Output is saved as: <your_pdf_name>_pageindex.json

import json

TREE_JSON_PATH = "/path/to/your/document_pageindex.json"  # ← change this

with open(TREE_JSON_PATH, "r") as f:
    local_tree = json.load(f)

print(f"🌲 Local tree loaded: {count_nodes(local_tree)} total nodes")
print_tree(local_tree)

In [ ]:
# ── Run the same RAG pipeline on the local tree ──────────────────────────────
# Everything from Sections 4–6 works identically with local trees

query  = "Summarize the executive summary section."
answer = vectorless_rag(query, local_tree)

---
## 📊 Section 9: Vector RAG vs PageIndex — Side-by-Side

### Architecture Comparison

| Aspect | Traditional Vector RAG | PageIndex (Vectorless RAG) |
|--------|------------------------|---------------------------|
| **Document prep** | Chunk into fixed pieces | Build hierarchical tree |
| **Indexing** | Embed each chunk | LLM reads structure |
| **Storage** | Vector database | JSON file |
| **Query processing** | Embed query → ANN search | LLM reasons over tree |
| **What's retrieved** | Flat anonymous chunks | Named sections + page refs |
| **Explainability** | ❌ Opaque similarity score | ✅ Traceable reasoning |
| **Domain expertise** | ❌ Needs embedding fine-tune | ✅ Add rules to prompt |
| **Infrastructure** | Pinecone / FAISS / ChromaDB | No vector DB needed |
| **Best for** | Short, diverse documents | Long, structured documents |
| **FinanceBench accuracy** | ~80% | **98.7%** |

### When to use which

**Use Vector RAG when:**
- Documents are short and varied (FAQs, product descriptions)
- Semantic paraphrase matching is important  
- You need sub-second retrieval on millions of documents

**Use PageIndex when:**
- Documents are long and professionally structured (reports, manuals, legal docs)
- You need traceable, cited answers
- Domain expertise should guide retrieval
- You want to avoid vector DB infrastructure


In [ ]:
# ── Quick comparison demo ────────────────────────────────────────────────────
# Show how the same query retrieves differently

print("=" * 55)
print("VECTOR RAG approach (conceptual):")
print("=" * 55)
print("""
query_vec = embed_model.encode("What are EBITDA risks?")
chunks    = vector_db.similarity_search(query_vec, k=5)

# Returns: 5 text fragments ranked by cosine distance
# Problem: may return "market risk" chunks, not EBITDA section
# No page numbers, no section context
""")

print("=" * 55)
print("PAGEINDEX approach (actual):")
print("=" * 55)
print("""
result = llm_tree_search("What are EBITDA risks?", tree)

# Returns: node IDs like ["0007", "0012"]
# LLM reasoning: "EBITDA is discussed in MD&A section (node 0007)
#                 and footnotes in Financial Statements (node 0012)"
# Full traceability — section title + page number
""")

---
## 🧹 Section 10: Cleanup

Delete documents from the PageIndex cloud when you're done  
to keep your storage clean.


In [ ]:
# ── Delete document from cloud ───────────────────────────────────────────────
# WARNING: This permanently deletes the tree index.
# Comment this out if you want to reuse the doc_id later.

# pi_client.delete_document(doc_id)
# print(f"🗑️ Deleted document: {doc_id}")
print("ℹ️ Deletion commented out — uncomment when you're done with this doc_id")

---
## ✅ Summary

You've now built a complete **Vectorless RAG** system with PageIndex.

### What you built:

1. **`llm_tree_search()`** — LLM reasons over document tree to find relevant nodes
2. **`find_nodes_by_ids()`** — Retrieve actual section content from tree
3. **`generate_answer()`** — LLM produces cited, grounded answers
4. **`vectorless_rag()`** — Full pipeline combining all 3 steps
5. **`expert_rag()`** — Domain-guided retrieval without any fine-tuning
6. **Chat API** — Zero-setup document Q&A

### Key takeaways:

- `Similarity ≠ Relevance` — the fundamental flaw of vector search
- Tree-based reasoning gives you **traceable**, **accurate**, **explainable** retrieval
- Domain expertise injection is just **prompt engineering** — no model training needed
- 98.7% on FinanceBench vs ~80% for vector RAG

---

### 🔗 Resources
- GitHub: https://github.com/VectifyAI/PageIndex
- Docs: https://docs.pageindex.ai
- Chat Platform: https://chat.pageindex.ai
- Blog: https://pageindex.ai/blog/pageindex-intro

---
*Crash course by [@krishnaik06](https://youtube.com/@krishnaik06) | [krishnaik.in/liveclasses](https://krishnaik.in/liveclasses)*
